# How to locate a buggy sub with subwise counting

## The problem

When a large FTQC circuit uses more T gates, more ancillas, or more Toffolis than you expected, the aggregate count from `GateCountEvaluatorHooks` tells you *that* something is off but not *where*. The usual workaround is to chop the circuit into pieces by hand, compile each piece, and count each piece in isolation until the offending sub shows up.

Subwise counting removes that detour. It walks the compiled `MachineSub` once and reports, per `Sub`:

- how many times the sub is called from the upper-level sub,
- how many qubits are required by the sub (input/output plus auxiliary qubits used inside it),
- a breakdown of the primitive gates contained directly in that sub and rolled up across everything it calls.

With that report in hand, the sub whose numbers do not match your manually calculated expectation is the sub to inspect.

This guide walks through that workflow end to end:

1. Build a parametric sub whose gate count should grow linearly with its parameter.
2. Sweep the parameter, observe the count exploding exponentially, and watch aggregate counting fail to localise the bug.
3. Run subwise counting, read the per-level report, and pinpoint the bad recursion.
4. Fix the sub and confirm with subwise counting again.

## Step 1. Build a parametric circuit with nested subs

Suppose you are writing your own decomposition of a multi-controlled X, `MCX(n)`, perhaps because your target gateset or ancilla budget rules out `std.MultiControlled` directly. You expect the Toffoli count to grow **linearly** in the number of controls `n`: roughly one Toffoli per added control.

We will build `MCX(n)` recursively: each level peels off one control and recurses on the rest. The **intended** body has one recursive call per level. The **buggy** body has a copy-paste that recurses twice — a classic mistake when hand-rolling reversible-arithmetic-style decompositions. Aggregate counts will tell us the Toffoli count doubles per added control; they will not tell us *where* the doubling comes from.

Note: the simplified decomposition below is not a correct MCX as a unitary — a real decomposition would borrow ancilla qubits. The tutorial point is gate-count debugging, not decomposition correctness, so we keep the structure stylised and focus on how the counts scale.

In [ ]:
from collections.abc import Sequence

from quri_parts.qsub.opsub import ParamUnitarySubDef, param_opsub
from quri_parts.qsub.register import QRegSpec
from quri_parts.qsub.sub import SubBuilder
from quri_parts.qsub.lib import std


class _MCX(ParamUnitarySubDef[int]):
    name = "MCX"

    def qubit_count_fn(self, n: int) -> int:
        return n + 1

    def qregs_fn(self, n: int) -> Sequence[QRegSpec]:
        return (QRegSpec("ctrl", n), QRegSpec("tgt", 1))

    def sub(self, b: SubBuilder, n: int) -> None:
        ctrls = b.qregs["ctrl"].qubits
        (tgt,) = b.qregs["tgt"].qubits
        if n == 1:
            b.add_op(std.CNOT, (ctrls[0], tgt))
            return
        if n == 2:
            b.add_op(std.Toffoli, (ctrls[0], ctrls[1], tgt))
            return
        # Intended (linear): one recursive call + one Toffoli per level.
        #   b.add_op(MCX(n - 1), (*ctrls[:-1], tgt))
        #   b.add_op(std.Toffoli, (ctrls[-2], ctrls[-1], tgt))
        # Bug: the recursive call was pasted twice. Toffoli(n) = 2 Toffoli(n-1) + 1.
        b.add_op(MCX(n - 1), (*ctrls[:-1], tgt))
        b.add_op(std.Toffoli, (ctrls[-2], ctrls[-1], tgt))
        b.add_op(MCX(n - 1), (*ctrls[:-1], tgt))


MCX, _ = param_opsub(_MCX)

## Step 2. Why aggregate counting is not enough

Sweep `n` and run the existing `GateCountEvaluatorHooks`. The Toffoli count should grow by one per added control. Instead it doubles.

In [ ]:
from quri_parts.qsub.compile import compile
from quri_parts.qsub.evaluate import Evaluator
from quri_parts.qsub.eval import GateCountEvaluatorHooks

primitives = (
    std.X, std.H, std.T, std.Tdag, std.S, std.Sdag,
    std.CNOT, std.Toffoli,
)

for n in range(2, 8):
    compiled = compile(MCX(n), primitives)
    totals = Evaluator(GateCountEvaluatorHooks((std.Toffoli,))).run(compiled)
    print(f"n={n}  ->", {k[1]: v for k, v in totals.items()})

Expected: `Toffoli = 1, 2, 3, 4, 5, 6`. Observed: `1, 3, 7, 15, 31, 63` — that is `2^{n-1} - 1`. From this table alone you cannot tell whether the doubling is in `MCX`'s own body, in the recursion structure, or in how the framework expands something downstream. Up to now the only way forward would be to recompile each candidate sub in isolation. Subwise counting replaces that loop.

## Step 3. Run subwise counting

`SubwiseCountEvaluatorHooks` is the new hook. Its API mirrors `GateCountEvaluatorHooks` so it slots into the same `Evaluator` pipeline:

- Constructor takes the gates you want broken down. Pass nothing to count *every* primitive that appears.
- `run(compiled)` returns one entry per distinct sub keyed by `base_id`, so two subs that happen to share a `name` cannot collide. For a parametric sub like `MCX`, each parameter value gets its own entry.
- Each entry exposes:
  - `name` — readable name.
  - `call_count` — how many times this sub is invoked across the program.
  - `qubit_count_info` — input/output width (`data`) and auxiliary qubits allocated inside the sub (`auxiliary`), plus their sum (`total`).
  - `gate_count_info.self_per_call` / `self` — gates contained *directly* in this sub, per single invocation and totalled across all invocations.
  - `gate_count_info.descendants_per_call` / `descendants` — same, rolled up across every sub this one calls into.
  - `gate_count_info.total_per_call` / `total` — `self + descendants`.

In [ ]:
import pandas as pd

# Dummy data — `SubwiseCountEvaluatorHooks` is not implemented yet.
# Once it lands, this cell becomes:
#   from quri_parts.qsub.eval import SubwiseCountEvaluatorHooks
#   compiled = compile(MCX(6), primitives)
#   report_data = Evaluator(SubwiseCountEvaluatorHooks(gates=(std.Toffoli,))).run(compiled)
report_data = [
    {
        "name": "MCX(6)",
        "call_count": 1,
        "qubit_count_info": {"auxiliary": 0, "data": 7, "total": 7},
        "gate_count_info": {
            "self_per_call": {"Toffoli": 1},
            "self": {"Toffoli": 1},
            "descendants_per_call": {"Toffoli": 30},
            "descendants": {"Toffoli": 30},
            "total_per_call": {"Toffoli": 31},
            "total": {"Toffoli": 31},
        },
    },
    {
        "name": "MCX(5)",
        "call_count": 2,
        "qubit_count_info": {"auxiliary": 0, "data": 6, "total": 6},
        "gate_count_info": {
            "self_per_call": {"Toffoli": 1},
            "self": {"Toffoli": 2},
            "descendants_per_call": {"Toffoli": 14},
            "descendants": {"Toffoli": 28},
            "total_per_call": {"Toffoli": 15},
            "total": {"Toffoli": 30},
        },
    },
    {
        "name": "MCX(4)",
        "call_count": 4,
        "qubit_count_info": {"auxiliary": 0, "data": 5, "total": 5},
        "gate_count_info": {
            "self_per_call": {"Toffoli": 1},
            "self": {"Toffoli": 4},
            "descendants_per_call": {"Toffoli": 6},
            "descendants": {"Toffoli": 24},
            "total_per_call": {"Toffoli": 7},
            "total": {"Toffoli": 28},
        },
    },
    {
        "name": "MCX(3)",
        "call_count": 8,
        "qubit_count_info": {"auxiliary": 0, "data": 4, "total": 4},
        "gate_count_info": {
            "self_per_call": {"Toffoli": 1},
            "self": {"Toffoli": 8},
            "descendants_per_call": {"Toffoli": 2},
            "descendants": {"Toffoli": 16},
            "total_per_call": {"Toffoli": 3},
            "total": {"Toffoli": 24},
        },
    },
    {
        "name": "MCX(2)",
        "call_count": 16,
        "qubit_count_info": {"auxiliary": 0, "data": 3, "total": 3},
        "gate_count_info": {
            "self_per_call": {"Toffoli": 1},
            "self": {"Toffoli": 16},
            "descendants_per_call": {},
            "descendants": {},
            "total_per_call": {"Toffoli": 1},
            "total": {"Toffoli": 16},
        },
    },
]

pd.set_option("display.max_colwidth", None)
df = pd.DataFrame(report_data)
df

Two things jump out from the table above:

1. `gate_count_info.self_per_call` is `{Toffoli: 1}` at every level. No single body is doing anything pathological — there is no level where 'too many' Toffolis are inlined.
2. `call_count` doubles per level (`1, 2, 4, 8, 16`) and `gate_count_info.descendants_per_call` doubles per level (`30, 14, 6, 2, 0`). The blow-up lives in the **child fan-out**, not in any one body.

That diagnosis — 'each level calls the next level too many times' — points the eye directly at the recursive section of `_MCX.sub`, not at the gate list. Aggregate counts could not distinguish 'too many Toffolis in one place' from 'recursion forks too often'.

## Step 4. Zoom in on the offending sub

Pick a single entry out of the report and inspect it on its own. With the real hook this would be `report.for_sub(MCX(5).base_id)`; on the dummy data we just filter the list.

In [ ]:
mcx5_entry = next(e for e in report_data if e["name"] == "MCX(5)")
print("name:", mcx5_entry["name"])
print("calls:", mcx5_entry["call_count"])
print("qubits:", mcx5_entry["qubit_count_info"])
print("self (per call):", mcx5_entry["gate_count_info"]["self_per_call"])
print("descendants (per call):", mcx5_entry["gate_count_info"]["descendants_per_call"])
print("total (per call):", mcx5_entry["gate_count_info"]["total_per_call"])

`MCX(5)` contains only one Toffoli of its own (`self_per_call = {Toffoli: 1}`), yet rolls up to 15 Toffolis per call once descendants are included. The factor-of-15 gap is the recursion misbehaving below it — not anything inside the `MCX(5)` body itself.

## Step 5. Fix and re-check

Delete the duplicate recursive call from `_MCX.sub`, redefine the sub, and sweep `n` again. The Toffoli count should now grow by one per added control.

In [ ]:
class _MCX(ParamUnitarySubDef[int]):  # noqa: F811 — replacing the buggy definition
    name = "MCX"

    def qubit_count_fn(self, n: int) -> int:
        return n + 1

    def qregs_fn(self, n: int) -> Sequence[QRegSpec]:
        return (QRegSpec("ctrl", n), QRegSpec("tgt", 1))

    def sub(self, b: SubBuilder, n: int) -> None:
        ctrls = b.qregs["ctrl"].qubits
        (tgt,) = b.qregs["tgt"].qubits
        if n == 1:
            b.add_op(std.CNOT, (ctrls[0], tgt))
            return
        if n == 2:
            b.add_op(std.Toffoli, (ctrls[0], ctrls[1], tgt))
            return
        b.add_op(MCX(n - 1), (*ctrls[:-1], tgt))
        b.add_op(std.Toffoli, (ctrls[-2], ctrls[-1], tgt))


MCX, _ = param_opsub(_MCX)

for n in range(2, 8):
    compiled = compile(MCX(n), primitives)
    totals = Evaluator(GateCountEvaluatorHooks((std.Toffoli,))).run(compiled)
    print(f"n={n}  ->", {k[1]: v for k, v in totals.items()})

Subwise counting on the fixed version should now show `call_count = 1` for every level and `descendants_per_call` shrinking by one per level. Dummy version of that report shown below for comparison with Step 3.

In [ ]:
report_data_fixed = [
    {
        "name": "MCX(6)",
        "call_count": 1,
        "qubit_count_info": {"auxiliary": 0, "data": 7, "total": 7},
        "gate_count_info": {
            "self_per_call": {"Toffoli": 1},
            "self": {"Toffoli": 1},
            "descendants_per_call": {"Toffoli": 4},
            "descendants": {"Toffoli": 4},
            "total_per_call": {"Toffoli": 5},
            "total": {"Toffoli": 5},
        },
    },
    {
        "name": "MCX(5)",
        "call_count": 1,
        "qubit_count_info": {"auxiliary": 0, "data": 6, "total": 6},
        "gate_count_info": {
            "self_per_call": {"Toffoli": 1},
            "self": {"Toffoli": 1},
            "descendants_per_call": {"Toffoli": 3},
            "descendants": {"Toffoli": 3},
            "total_per_call": {"Toffoli": 4},
            "total": {"Toffoli": 4},
        },
    },
    {
        "name": "MCX(4)",
        "call_count": 1,
        "qubit_count_info": {"auxiliary": 0, "data": 5, "total": 5},
        "gate_count_info": {
            "self_per_call": {"Toffoli": 1},
            "self": {"Toffoli": 1},
            "descendants_per_call": {"Toffoli": 2},
            "descendants": {"Toffoli": 2},
            "total_per_call": {"Toffoli": 3},
            "total": {"Toffoli": 3},
        },
    },
    {
        "name": "MCX(3)",
        "call_count": 1,
        "qubit_count_info": {"auxiliary": 0, "data": 4, "total": 4},
        "gate_count_info": {
            "self_per_call": {"Toffoli": 1},
            "self": {"Toffoli": 1},
            "descendants_per_call": {"Toffoli": 1},
            "descendants": {"Toffoli": 1},
            "total_per_call": {"Toffoli": 2},
            "total": {"Toffoli": 2},
        },
    },
    {
        "name": "MCX(2)",
        "call_count": 1,
        "qubit_count_info": {"auxiliary": 0, "data": 3, "total": 3},
        "gate_count_info": {
            "self_per_call": {"Toffoli": 1},
            "self": {"Toffoli": 1},
            "descendants_per_call": {},
            "descendants": {},
            "total_per_call": {"Toffoli": 1},
            "total": {"Toffoli": 1},
        },
    },
]

pd.DataFrame(report_data_fixed)

Production code would normally reach for `std.MultiControlled` — that is the audited reference. Subwise counting is what lets you audit your own subs when you have to roll one.